# Mapa de proyectos y mejoras probables de examen

Esta notebook no es teoría. Es una guía para decidir qué proyectos merece la pena llevar preparados al examen y qué mejoras pueden pedir.

Criterio usado:

- si funciona sin internet una vez instaladas dependencias
- si el dataset está dentro del repo o requiere descarga
- si el proyecto tiene comentarios de mejora, retos, TODOs o conclusiones abiertas
- si la mejora es razonable para un examen práctico


## Resumen rápido

| Proyecto | Prioridad | Offline en examen | Veredicto |
|---|---:|---|---|
| `california-e2e` | Alta | Sí, trae `data/housing.csv` | Muy probable: regresión tabular end to end |
| `king-county` | Alta si descargas datos antes | No trae CSV en repo | Probable si el dataset está cacheado o descargado |
| `thyroid` | Media/alta si descargas datos antes | No trae CSV en repo | Muy buen candidato de clasificación médica |
| `ai-chat-guardrails` | Media | Solo con Ollama ya instalado/modelo descargado | Puede caer como mejora de tests/arquitectura, no IA remota |
| `computer-vision` | Media/baja | OpenCV sí; YOLO/CLIP no sin modelos | Útil para OpenCV; descartar partes con modelos si no están descargados |
| `IES_Teis_AI_Triage` | Baja | Requiere Ollama, embeddings y/o Chroma precreado | Solo si dejan todo preparado; si no, descartarlo |


## Proyectos que yo priorizaría

1. `california-e2e`

Es el más seguro: dataset local, regresión clásica, pipelines, imputación, escalado, categóricas, búsqueda de hiperparámetros y red neuronal. Es exactamente el tipo de proyecto donde el profesor puede pedir “mejora este modelo” o “evita fuga de datos”.

2. `king-county`

Muy buena pinta para examen porque tiene problemas reales: IDs repetidos, fuga temporal, división cronológica, feature engineering, pipeline y deep learning. El problema: no trae CSV en el repo; usa Kaggle. Hay que descargar/cachear antes.

3. `thyroid`

Muy buen proyecto de clasificación: desbalance, valores faltantes informativos, métricas clínicas, RandomForest, XGBoost y red neuronal. El problema: tampoco trae CSV en el repo; usa Kaggle.


## Proyectos que descartaría o miraría solo por encima

### `IES_Teis_AI_Triage`

Tiene un reto explícito:

- procesar nuevos correos sin depender de un JSON con nombre fijo
- pasar de RAG básico a RAG 2.0

Pero para examen offline es arriesgado porque depende de:

- Ollama ejecutándose
- modelo Llama descargado
- embeddings de Hugging Face descargados
- base Chroma creada
- librerías LangChain funcionando

Solo lo prepararía si antes del examen dejas todo instalado, modelos descargados y una base vectorial ya persistida.

### `computer-vision`

OpenCV básico sí es viable offline porque usa imágenes locales. Pero YOLO/CLIP/face recognition son peligrosos sin preparación previa:

- YOLO descarga pesos en primera ejecución
- CLIP descarga un modelo grande
- algunos ejemplos necesitan webcam o vídeos externos

Úsalo para preguntas de procesamiento de imagen clásico; no confiaría en YOLO/CLIP salvo que tengas pesos ya descargados.

### `ai-chat-guardrails`

Gemini queda descartado sin internet/API. Ollama solo sirve si ya tienes Ollama instalado, servidor activo y modelo descargado.

Aun así, puede caer una mejora de código sin ejecutar IA real: tests, guardrails, historial, configuración, mocks.


## Mejoras probables: `california-e2e`

Candidatos muy probables:

- añadir o modificar ingeniería de características: ratios, log transform, clusters geográficos
- evitar data leakage en preprocesamiento
- convertir pasos sueltos en `Pipeline` o `ColumnTransformer`
- ajustar hiperparámetros con `GridSearchCV` o `RandomizedSearchCV`
- comparar modelos con validación cruzada
- mejorar la red neuronal: más/menos capas, dropout, early stopping, target scaling
- invertir el escalado de la variable objetivo para interpretar RMSE en dólares
- tratar valores faltantes correctamente con `SimpleImputer`
- explicar por qué `median_income` se estratifica en el split

Frase clave para examen:

“Primero separo train/test, luego ajusto imputadores, escaladores y encoders solo con train usando un pipeline para evitar leakage.”


## Mejoras probables: `king-county`

Candidatos muy probables:

- cambiar split aleatorio por split temporal
- eliminar `id` como feature directa
- tratar IDs repetidos como reventas, no como duplicados inválidos
- extraer año/mes de `date`
- crear features como antigüedad de la casa, renovación, ratios de superficie
- usar lat/lon o clustering geográfico
- guardar/reutilizar pipeline de preprocesado
- comparar baseline, Ridge/Lasso, RandomForest, XGBoost
- añadir early stopping a la red neuronal
- comparar RMSE/MAE/R2 entre modelo clásico y deep learning

Cuidado grande:

Si el dataset no está descargado antes, este proyecto puede quedarse bloqueado porque usa `kagglehub.dataset_download`.


## Mejoras probables: `thyroid`

Candidatos muy probables:

- adaptar el preprocesamiento según modelo: lineales, árboles, red neuronal
- mejorar recall de enfermedades, especialmente hiper/hipotiroidismo
- usar `class_weight`, `sample_weight` o pesos en `CrossEntropyLoss`
- explicar por qué accuracy es mala métrica con clases desbalanceadas
- usar F2 Score en lugar de accuracy/F1 normal
- convertir edades imposibles en NaN
- decidir qué hacer con `TBG` y columnas `_measured`
- comparar RandomForest vs XGBoost vs red neuronal
- ajustar umbral o coste de error para reducir falsos negativos

Cuidado offline:

El loader descarga desde Kaggle. Conviene tener `thyroidDF.csv` descargado o cacheado antes del examen.


## Mejoras probables: `ai-chat-guardrails`

Este proyecto tiene mejoras bastante claras aunque no puedas llamar a Gemini/Ollama.

Encontrado al ejecutar tests:

- `uv run pytest` falla porque `chatbot` no se descubre como paquete si no se usa `PYTHONPATH=.`
- con `PYTHONPATH=.` los tests corren, pero fallan 5 tests porque los mensajes esperados están en inglés y el código devuelve mensajes en español

Mejoras realistas de examen:

- corregir packaging o configuración de pytest para que `uv run pytest` funcione sin `PYTHONPATH=.`, por ejemplo configurando `pythonpath` en pytest
- alinear tests con mensajes reales en español, o hacer asserts menos frágiles
- arreglar el recorte de historial: el test dice máximo 4 mensajes, pero espera 5; hay una incoherencia entre comportamiento y expectativa
- hacer que `OllamaBackend` use `base_url`, porque la config lo define pero el factory no lo pasa
- añadir persistencia de historial en JSON
- añadir detección de PII o emails/teléfonos en guardrails
- añadir interfaz Gradio o FastAPI, si las dependencias están disponibles

Este es buen proyecto si el examen pide “mejora arquitectura/tests”, no si exige usar IA online.


## Mejoras probables: `IES_Teis_AI_Triage`

El propio `challenge.md` marca dos mejoras:

1. Gestionar nuevos correos sin depender de un fichero JSON fijo.
2. Mejorar el RAG hacia RAG 2.0.

Qué podrían pedir de forma razonable:

- leer todos los `.json` de una carpeta de entrada
- mover correos procesados a una carpeta `processed/`
- guardar errores en `failed/`
- evitar reprocesar correos ya respondidos
- añadir IDs únicos por correo
- separar correos respondidos/escalados en CSVs
- implementar MultiQueryRetriever
- añadir reranking si el modelo está descargado

Qué descartaría si no hay modelos descargados:

- reranker de Hugging Face desde cero
- descargar embeddings
- llamar a Llama/Ollama si no está activo
- crear Chroma desde cero si depende de modelos no cacheados


## Mejoras probables: `computer-vision`

Parte viable offline:

- OpenCV fundamentals
- procesamiento de imagen: blur, Canny, thresholding, morfología, contornos
- leer/escribir imágenes locales
- comparar BGR/RGB
- detectar bordes o contornos sobre imágenes de `resources/images`

Parte arriesgada offline:

- YOLO si no están los pesos `yolov8n.pt` descargados
- CLIP/SentenceTransformer si el modelo no está cacheado
- webcam si el entorno no tiene cámara o GUI
- vídeos externos descargados por URL

Mejoras razonables:

- añadir comprobaciones de calidad de imagen: brillo, desenfoque, tamaño mínimo
- guardar outputs en `artifacts/outputs`
- parametrizar umbrales de Canny/thresholding
- validar si una imagen se cargó antes de procesarla
- evitar que notebooks fallen si no hay webcam


## Checklist antes del examen sin internet

Instalar dependencias al principio:

- `california-e2e`: pandas, numpy, sklearn, matplotlib, seaborn, xgboost, torch si vas a deep learning
- `king-county`: lo mismo + descargar/cachear dataset de Kaggle
- `thyroid`: lo mismo + descargar/cachear dataset de Kaggle
- `ai-chat-guardrails`: instalar deps y, si quieres modo local, `ollama pull llama3.2`
- `computer-vision`: instalar deps y descargar pesos/modelos si vas a YOLO/CLIP
- `IES_Teis_AI_Triage`: instalar LangChain, descargar modelo Ollama, descargar embeddings y crear Chroma antes

Archivos/datos que deberían existir antes:

- `proyects/california-e2e/data/housing.csv` ya existe
- King County: `kc_house_data.csv` no está en repo, hay que tenerlo cacheado/descargado
- Thyroid: `thyroidDF.csv` no está en repo, hay que tenerlo cacheado/descargado
- YOLO: pesos `.pt` si vas a usarlo
- CLIP/SentenceTransformer: modelo cacheado si vas a embeddings
